In [1]:
!pip install firecrawl-py lancedb tantivy

In [2]:
!pip install langchain-text-splitters

How the document was built (scraped https://abdiadama.com/)

In [8]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("FIRECRAWL_API_KEY")

In [11]:
from firecrawl import Firecrawl
from firecrawl.v2.types import ScrapeOptions

app = Firecrawl(api_key=api_key)

scrape_opts = ScrapeOptions(
    only_main_content=False,
    max_age=172800000,
    parsers=["pdf"],
    formats=["markdown"]
)

crawl_result = app.crawl(
    "www.abdiadama.com/",
    sitemap="include",
    crawl_entire_domain=False,
    limit=10,
    scrape_options=scrape_opts
)

print(crawl_result)

status='completed' total=7 completed=7 credits_used=7 expires_at=datetime.datetime(2026, 5, 20, 13, 49, 22, tzinfo=TzInfo(0)) next=None data=[Document(markdown='[_Abdi Adama_ School](http://www.abdiadama.com/index.html)\n\n[menu](http://www.abdiadama.com/#menu "menu")\n\n- [Home](http://www.abdiadama.com/index.html)\n- [About Us](http://www.abdiadama.com/about-us.html)\n  - [What Makes Us Stand Out](http://www.abdiadama.com/about-us.html#about-us)\n  - [Mission & Vision](http://www.abdiadama.com/about-us.html#mission-vision)\n  - [Our Values](http://www.abdiadama.com/about-us.html#about-us)\n  - [Our Promise to Parents](http://www.abdiadama.com/about-us.html#promise)\n- [Our School](http://www.abdiadama.com/ourschool.html)\n- [Admissions](http://www.abdiadama.com/admissions.html)\n- [Careers](http://www.abdiadama.com/careers.html)\n- [Contact Us](http://www.abdiadama.com/contact-us.html)\n\n![](http://www.abdiadama.com/assets/images/schoolBuilding.jpg)\n\n###### Welcome To\n\n## _Abdi 

In [12]:
for doc in crawl_result.data:
  print(doc.markdown)
scraped_docs = crawl_result.data

[_Abdi Adama_ School](http://www.abdiadama.com/index.html)

[menu](http://www.abdiadama.com/#menu "menu")

- [Home](http://www.abdiadama.com/index.html)
- [About Us](http://www.abdiadama.com/about-us.html)
  - [What Makes Us Stand Out](http://www.abdiadama.com/about-us.html#about-us)
  - [Mission & Vision](http://www.abdiadama.com/about-us.html#mission-vision)
  - [Our Values](http://www.abdiadama.com/about-us.html#about-us)
  - [Our Promise to Parents](http://www.abdiadama.com/about-us.html#promise)
- [Our School](http://www.abdiadama.com/ourschool.html)
- [Admissions](http://www.abdiadama.com/admissions.html)
- [Careers](http://www.abdiadama.com/careers.html)
- [Contact Us](http://www.abdiadama.com/contact-us.html)

![](http://www.abdiadama.com/assets/images/schoolBuilding.jpg)

###### Welcome To

## _Abdi Adama_ School

###### To build the future of our children, we must plant the seeds of excellence today. A thriving  international  learning community   Supported by passionate educ

In [17]:
large_text = ""
for doc in scraped_docs: 
    if doc.markdown:
        cleaned_text = doc.markdown
        large_text += cleaned_text + "\n\n---\n\n" 

In [18]:
large_text

'[_Abdi Adama_ School](http://www.abdiadama.com/index.html)\n\n[menu](http://www.abdiadama.com/#menu "menu")\n\n- [Home](http://www.abdiadama.com/index.html)\n- [About Us](http://www.abdiadama.com/about-us.html)\n  - [What Makes Us Stand Out](http://www.abdiadama.com/about-us.html#about-us)\n  - [Mission & Vision](http://www.abdiadama.com/about-us.html#mission-vision)\n  - [Our Values](http://www.abdiadama.com/about-us.html#about-us)\n  - [Our Promise to Parents](http://www.abdiadama.com/about-us.html#promise)\n- [Our School](http://www.abdiadama.com/ourschool.html)\n- [Admissions](http://www.abdiadama.com/admissions.html)\n- [Careers](http://www.abdiadama.com/careers.html)\n- [Contact Us](http://www.abdiadama.com/contact-us.html)\n\n![](http://www.abdiadama.com/assets/images/schoolBuilding.jpg)\n\n###### Welcome To\n\n## _Abdi Adama_ School\n\n###### To build the future of our children, we must plant the seeds of excellence today. A thriving  international  learning community   Suppor

Or you can just read a large .txt containing the full data

In [20]:
with open('data.txt', 'r') as file:
    content = file.read()

print(content)
large_text = content

What Sets Abdi Adama School Apart?
At Abdi Adama School, we don’t just teach; we transform. We pride ourselves on a distinct educational philosophy built upon four core pillars that guide our culture and inspire our students to become impactful global citizens.
The Four Pillars of Our Excellence
Pillar	Our Commitment
Integrity	Doing what is right, always. We foster honesty and accountability in every interaction.
Leadership	Cultivating the potential within. We empower students to take initiative and inspire others.
Success	A holistic mindset. We define success by resilience and the pursuit of one's full potential.
Lifelong Learning	Staying curious. We equip students with the agility to thrive in a constantly evolving world.
1. Integrity: Our Moral Compass
Integrity is the heartbeat of our community. We believe that character is defined by what you do when no one is watching. By upholding the highest ethical standards, we ensure our students graduate with a grounded sense of transparenc

Chunking

In [23]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter


def clean_markdown(text):
    if not text:
        return ""
    
    # Remove extra blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()


print(f"Total size of large text: {len(large_text)} characters")

# Chunk the Large Text using LangChain
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # Max characters per chunk
    chunk_overlap=200,     # Overlap 
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""] # Order of preference for splitting
)

# Generate the chunks
chunks = text_splitter.create_documents([large_text])

# Print results
print(f"Created {len(chunks)} chunks.")
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)

Total size of large text: 15189 characters
Created 23 chunks.

--- Chunk 1 ---
What Sets Abdi Adama School Apart?
At Abdi Adama School, we don’t just teach; we transform. We pride ourselves on a distinct educational philosophy built upon four core pillars that guide our culture and inspire our students to become impactful global citizens.
The Four Pillars of Our Excellence
Pillar	Our Commitment
Integrity	Doing what is right, always. We foster honesty and accountability in every interaction.
Leadership	Cultivating the potential within. We empower students to take initiative and inspire others.
Success	A holistic mindset. We define success by resilience and the pursuit of one's full potential.
Lifelong Learning	Staying curious. We equip students with the agility to thrive in a constantly evolving world.
1. Integrity: Our Moral Compass

--- Chunk 2 ---
Lifelong Learning	Staying curious. We equip students with the agility to thrive in a constantly evolving world.
1. Integrity: Our Moral Co

In [24]:
!pip install lancedb sentence-transformers pandas pydantic

#Indexing

In [36]:
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry
import pandas as pd

# 1. Setup the Embedding Model
embedder = get_registry().get("sentence-transformers").create(name="BAAI/bge-small-en-v1.5")

# 2. Define the Schema
class SchoolDocument(LanceModel):
    text: str = embedder.SourceField()
    vector: Vector(embedder.ndims()) = embedder.VectorField()
    chunk_id: int

# 3. Connect to LanceDB
db = lancedb.connect("./lancedb_school_data")
table_name = "abdi_adama_docs"

# 4. Safely Open or Create the Table
if table_name in db.table_names():
    table = db.open_table(table_name)
    print(f"Opened existing table '{table_name}'. New data will be appended.")
else:
    table = db.create_table(table_name, schema=SchoolDocument)
    print(f"Table '{table_name}' not found. Created a new table successfully!")

# 5. Prepare the Data to Insert
data_to_insert = []
for idx, chunk in enumerate(chunks): 
    data_to_insert.append({
        "text": chunk.page_content,
        "chunk_id": idx
    })

# 6. Embed and Append Data
print(f"Embedding and appending {len(data_to_insert)} chunks into LanceDB... (This may take a few seconds)")
table.add(data_to_insert)  
print("Data successfully appended!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\yosep\AppData\Local\Temp\ipykernel_16140\1120629128.py:20: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  if table_name in db.table_names():


Table 'abdi_adama_docs' not found. Created a new table successfully!
Embedding and appending 23 chunks into LanceDB... (This may take a few seconds)
Data successfully appended!


In [37]:
table = db.open_table("abdi_adama_docs")

In [38]:
user_query = "what's unique about the school?"

print(f"\nSearching for: '{user_query}'\n")

results = table.search(user_query).limit(3).to_pandas()

# Display the results
for index, row in results.iterrows():
    print(f"--- Result {index + 1} (Chunk ID: {row['chunk_id']}, Distance: {row['_distance']:.4f}) ---")
    print(row['text'])
    print("-" * 80)


Searching for: 'what's unique about the school?'

--- Result 1 (Chunk ID: 7, Distance: 0.7416) ---
• Welcomed: We invite you to be an active participant in school life through events, meetings, and initiatives.
• True Partners: Working hand-in-hand with us to nurture your child’s academic and personal evolution.
🎓 Our Teachers: Mentors, Role Models, and Visionaries
Our educators are more than subject matter experts; they are the architects of our students' futures. We take pride in a faculty that is:
• Values-Driven: Operating with deep empathy, integrity, and an unwavering focus on student success.
• Dynamic & Enthusiastic: Bringing energy to the classroom that makes learning infectious.
• Progressive & Innovative: Utilizing 21st-century methodologies and technologies to stay ahead of the curve.
• Deeply Caring: Building the trusting relationships and pastoral support necessary for student well-being.
• Ambitious: Setting high benchmarks for themselves and their students, never settl